Import the libraries

In [1]:
import json
import numpy as np

from pathlib import Path
from sentence_transformers import SentenceTransformer

print("Libraries imported successfully!")

Libraries imported successfully!


Load your chunked documents

In [2]:
PROJECT_ROOT = Path.cwd().parent

INPUT_FILE = (
    PROJECT_ROOT
    / "knowledge_base"
    / "processed"
    / "chunked_documents.jsonl"
)

chunked_documents = []

with open(INPUT_FILE, "r", encoding="utf-8") as file:
    for line in file:
        chunked_documents.append(json.loads(line))

print("Total chunks:", len(chunked_documents))

Total chunks: 1017


Load the embedding model

In [3]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


Generate embeddings

In [4]:
texts = [
    chunk["text"]
    for chunk in chunked_documents
]

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings generated successfully!")
print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Embeddings generated successfully!
Embedding shape: (1017, 384)


Save the embeddings

In [5]:
OUTPUT_FOLDER = (
    PROJECT_ROOT
    / "knowledge_base"
    / "processed"
)

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

EMBEDDING_FILE = OUTPUT_FOLDER / "chunk_embeddings.npy"

np.save(EMBEDDING_FILE, embeddings)

print("Embeddings saved:", EMBEDDING_FILE)

Embeddings saved: d:\FactoryMindAI\knowledge_base\processed\chunk_embeddings.npy


Validate your embeddings

In [6]:
saved_embeddings = np.load(EMBEDDING_FILE)

print("Total chunks:", len(chunked_documents))
print("Embedding count:", len(saved_embeddings))
print("Vector dimensions:", saved_embeddings.shape[1])
print("All values finite:", np.isfinite(saved_embeddings).all())

assert len(chunked_documents) == len(saved_embeddings)
assert saved_embeddings.shape[1] == 384
assert np.isfinite(saved_embeddings).all()

print("Embedding validation successful!")

Total chunks: 1017
Embedding count: 1017
Vector dimensions: 384
All values finite: True
Embedding validation successful!


Test semantic similarity

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

questions = [
    "How should machine bearings be inspected?",
    "What are the procedures for bearing inspection?",
    "What is the best way to prepare a vegetable salad?"
]

question_embeddings = embedding_model.encode(
    questions,
    normalize_embeddings=True
)

similarity = cosine_similarity(
    question_embeddings,
    question_embeddings
)

print(np.round(similarity, 3))

[[1.    0.837 0.135]
 [0.837 1.    0.156]
 [0.135 0.156 1.   ]]
